In [1]:
import sys, platform, subprocess
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Git commit:", subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip())

Python: 3.7.17
Platform: Linux-7.0.0-31-generic-x86_64-with-debian-trixie-sid
Git commit: d10f28466a92ad5ba21566a045d51b01bc77ac46


In [2]:
import pandas as pd

DATA_PATH = Path("..") / "assets" / "data" / "Fish.csv"
df = pd.read_csv(DATA_PATH)

print("shape:", df.shape)
print("columns:", list(df.columns))
df.head()

shape: (159, 7)
columns: ['Species', 'Weight', 'Length1', 'Length2', 'Length3', 'Height', 'Width']


,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340


In [3]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(dropna=False),
    "min": df.select_dtypes("number").min(),
    "max": df.select_dtypes("number").max(),
})
quality

,dtype,missing,unique,min,max
Height,float64,0,154,1.7284,18.957
Length1,float64,0,116,7.5000,59.000
Length2,float64,0,93,8.4000,63.400
Length3,float64,0,124,8.8000,68.000
Species,object,0,7,NaN,NaN
Weight,float64,0,101,0.0000,1650.000
Width,float64,0,152,1.0476,8.142


In [4]:
df["Species"].value_counts()

Perch        56
Bream        35
Roach        20
Pike         17
Smelt        14
Parkki       11
Whitefish     6
Name: Species, dtype: int64

In [5]:
# --- Автоматические проверки качества (шаг 3.3) ---
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))

import importlib
import data_checks as dc
importlib.reload(dc)

report = dc.run_all_checks(df)

In [6]:
print("ФОРМА:")
print(report["shape"])

print("\nТИПЫ:")
print(report["dtypes"])

print("\nДУБЛИКАТЫ:")
print(report["duplicates"])

print("\nПРОПУСКИ:")
print(report["missing"])

print("\nБЕСКОНЕЧНОСТИ:")
print(report["infinities"])

print("\nНЕВОЗМОЖНЫЕ ЗНАЧЕНИЯ (<= 0):")
for col, info in report["positivity"].items():
    print(f"  {col}: n_violations={info['n_violations']}, min={info['min_value']}, indices={info['indices']}")

print("\nСОГЛАСОВАННОСТЬ ДЛИН (Length1 <= Length2 <= Length3):")
print(report["length_order"])

print("\nСОГЛАСОВАННОСТЬ Height/Width:")
print(report["volume_consistency"])

print("\nРАСПРЕДЕЛЕНИЕ ПО ВИДАМ:")
print(report["species_dist"])

print("\nРЕДКИЕ КАТЕГОРИИ (<10%):")
print(report["rare_categories"])

print("\nДИСБАЛАНС ВИДОВ:")
print(report["imbalance"])

print("\nПОЧТИ ОДИНАКОВЫЕ СТРОКИ:")
print(report["near_duplicates"])

print("\nПОЛЯ-КАНДИДАТЫ В ID:")
print(report["id_like"])

ФОРМА:
{'check': 'shape', 'rows': 159, 'cols': 7, 'rows_ok': True, 'cols_ok': True}

ТИПЫ:
           dtype
Species   object
Weight   float64
Length1  float64
Length2  float64
Length3  float64
Height   float64
Width    float64

ДУБЛИКАТЫ:
{'check': 'duplicates', 'subset': None, 'n_duplicated_rows': 0, 'duplicated_indices': []}

ПРОПУСКИ:
         missing  missing_pct
Species        0          0.0
Weight         0          0.0
Length1        0          0.0
Length2        0          0.0
Length3        0          0.0
Height         0          0.0
Width          0          0.0

БЕСКОНЕЧНОСТИ:
         pos_inf  neg_inf
Weight         0        0
Length1        0        0
Length2        0        0
Length3        0        0
Height         0        0
Width          0        0

НЕВОЗМОЖНЫЕ ЗНАЧЕНИЯ (<= 0):
  Weight: n_violations=1, min=0.0, indices=[40]
  Length1: n_violations=0, min=7.5, indices=[]
  Length2: n_violations=0, min=8.4, indices=[]
  Length3: n_violations=0, min=8.8, indices=[]
  H

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

FIG_DIR = Path("..") / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 130
plt.rcParams["savefig.bbox"] = "tight"

def save_fig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path)
    print(f"Сохранено: {path.resolve()}")

print("FIG_DIR:", FIG_DIR.resolve())
print("Файлов в папке:", len(list(FIG_DIR.glob('*.png'))))

FIG_DIR: /home/egor/lab_2_semester/python/AI/fishgrow/reports/figures
Файлов в папке: 0


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(df["Weight"], bins=25, color="steelblue", edgecolor="black")
axes[0].set_title("Распределение массы рыбы (исходная шкала)")
axes[0].set_xlabel("Масса, г")
axes[0].set_ylabel("Число наблюдений")
axes[0].axvline(df["Weight"].mean(),   color="red",    linestyle="--",
                label=f"среднее = {df['Weight'].mean():.0f} г")
axes[0].axvline(df["Weight"].median(), color="orange", linestyle="--",
                label=f"медиана = {df['Weight'].median():.0f} г")
axes[0].legend()

w_pos = df["Weight"].clip(lower=1e-3)
axes[1].hist(np.log(w_pos), bins=25, color="seagreen", edgecolor="black")
axes[1].set_title("Распределение логарифма массы")
axes[1].set_xlabel("ln(масса, г)")
axes[1].set_ylabel("Число наблюдений")

fig.suptitle("Масса рыбы: исходная и логарифмическая шкала (n=159)", y=1.02)
plt.tight_layout()
save_fig(fig, "fig01_weight_distribution.png")
plt.show()

print("\n=== Интерпретация ===")
print(f"Среднее: {df['Weight'].mean():.0f} г")
print(f"Медиана: {df['Weight'].median():.0f} г")
print(f"Стандартное отклонение: {df['Weight'].std():.0f} г")
print(f"Асимметрия (skew): {df['Weight'].skew():.2f}")
print(f"Эксцесс (kurtosis): {df['Weight'].kurt():.2f}")
print("Вывод: распределение массы правосторонне асимметричное (тяжёлый правый хвост).")
print("Логарифмирование приближает распределение к симметричному.")

Сохранено: /home/egor/lab_2_semester/python/AI/fishgrow/reports/figures/fig01_weight_distribution.png

=== Интерпретация ===
Среднее: 398 г
Медиана: 273 г
Стандартное отклонение: 358 г
Асимметрия (skew): 1.10
Эксцесс (kurtosis): 0.88
Вывод: распределение массы правосторонне асимметричное (тяжёлый правый хвост).
Логарифмирование приближает распределение к симметричному.


/home/egor/lab_2_semester/python/AI/fishgrow/env/lib/python3.7/site-packages/ipykernel_launcher.py:22: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = df["Species"].value_counts().index.tolist()
counts = df["Species"].value_counts()

axes[0].barh(order, counts.values, color="steelblue", edgecolor="black")
axes[0].set_title("Численность по видам")
axes[0].set_xlabel("Число наблюдений")
axes[0].set_ylabel("Вид рыбы")
for i, v in enumerate(counts.values):
    axes[0].text(v + 0.5, i, str(v), va="center", fontsize=9)

sns.boxplot(data=df, y="Species", x="Weight", order=order, ax=axes[1], palette="Set2")
axes[1].set_title("Распределение массы по видам")
axes[1].set_xlabel("Масса, г")
axes[1].set_ylabel("Вид рыбы")

fig.suptitle("Численность и масса по видам рыб", y=1.02)
plt.tight_layout()
save_fig(fig, "fig02_weight_by_species.png")
plt.show()

print("\n=== Интерпретация ===")
print("Численность видов (дисбаланс):")
print(counts.to_string())
print(f"\nОтношение большинство/меньшинство: {counts.max()}/{counts.min()} = {counts.max()/counts.min():.2f}")
print("\nМедианная масса по видам, г:")
print(df.groupby("Species")["Weight"].median().sort_values(ascending=False).round(0).to_string())
print("\nВывод: 7 видов, сильный дисбаланс (Perch > Whitefish в ~9 раз).")
print("Виды различаются по массе: Pike и Whitefish тяжелее, Smelt — легче всех.")
print("Дисбаланс учитывается при разбиении (§3.7), стратифицированный split обязателен.")

Сохранено: /home/egor/lab_2_semester/python/AI/fishgrow/reports/figures/fig02_weight_by_species.png

=== Интерпретация ===
Численность видов (дисбаланс):
Perch        56
Bream        35
Roach        20
Pike         17
Smelt        14
Parkki       11
Whitefish     6

Отношение большинство/меньшинство: 56/6 = 9.33

Медианная масса по видам, г:
Species
Bream        610.0
Pike         510.0
Whitefish    423.0
Perch        208.0
Roach        148.0
Parkki       145.0
Smelt         10.0

Вывод: 7 видов, сильный дисбаланс (Perch > Whitefish в ~9 раз).
Виды различаются по массе: Pike и Whitefish тяжелее, Smelt — легче всех.
Дисбаланс учитывается при разбиении (§3.7), стратифицированный split обязателен.


/home/egor/lab_2_semester/python/AI/fishgrow/env/lib/python3.7/site-packages/ipykernel_launcher.py:21: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.


In [11]:
selected = ["Weight", "Length1", "Length2", "Length3", "Height"]
pair = sns.pairplot(
    df[selected + ["Species"]].dropna(),
    hue="Species",
    corner=False,
    height=1.7,
    plot_kws={"s": 18, "alpha": 0.7},
)
pair.fig.suptitle("Парные графики: масса и геометрические размеры (цвет — вид)", y=1.02)
pair.savefig(FIG_DIR / "fig03_pairplot.png", dpi=110)
print(f"Сохранено: {(FIG_DIR / 'fig03_pairplot.png').resolve()}")
plt.show()

print("\n=== Интерпретация ===")
print("Length1, Length2, Length3 практически коллинеарны —")
print("это три способа измерения одной и той же длины.")
print("Weight растёт с длинами, но связь визуально нелинейная (выпуклая).")
print("Наклоны регрессий отличаются между видами:")
print("для мелких видов (Smelt, Parkki) масса растёт медленнее,")
print("для крупных (Pike, Bream) — быстрее.")
print("Это означает, что вид — важный фактор связи «длина-масса».")

Сохранено: /home/egor/lab_2_semester/python/AI/fishgrow/reports/figures/fig03_pairplot.png

=== Интерпретация ===
Length1, Length2, Length3 практически коллинеарны —
это три способа измерения одной и той же длины.
Weight растёт с длинами, но связь визуально нелинейная (выпуклая).
Наклоны регрессий отличаются между видами:
для мелких видов (Smelt, Parkki) масса растёт медленнее,
для крупных (Pike, Bream) — быстрее.
Это означает, что вид — важный фактор связи «длина-масса».


/home/egor/lab_2_semester/python/AI/fishgrow/env/lib/python3.7/site-packages/ipykernel_launcher.py:12: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  if sys.path[0] == "":


In [12]:
numeric_cols = ["Weight", "Length1", "Length2", "Length3", "Height", "Width"]
corr = df[numeric_cols].corr(method="pearson")

fig, ax = plt.subplots(figsize=(7.5, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".3f", cmap="RdBu_r",
    vmin=-1, vmax=1, square=True, linewidths=0.5,
    cbar_kws={"label": "Коэффициент корреляции Пирсона"},
    ax=ax,
)
ax.set_title("Корреляционная матрица числовых признаков (Пирсон)")
plt.tight_layout()
save_fig(fig, "fig04_correlation.png")
plt.show()

print("\n=== Интерпретация ===")
print("Корреляции с целевой переменной (Weight):")
print(corr["Weight"].drop("Weight").sort_values(ascending=False).round(3).to_string())
print("\nТоп-3 пары признаков по |r|:")
pairs = corr.where(~np.eye(len(corr), dtype=bool)).unstack().dropna().sort_values(ascending=False)
print(pairs.drop_duplicates().head(3).round(3).to_string())
print("\nОговорка про нелинейные связи:")
print("Корреляция Пирсона измеряет ЛИНЕЙНУЮ связь. Если связь нелинейная")
print("(как Weight ~ Length здесь), коэффициент недооценивает силу связи.")
print("Например, r(Weight, Length) ≈ 0.9, но реальная зависимость ближе")
print("к степенной (W ~ L^3). Для линейной модели полезно логировать или")
print("строить признак Volume = Length3 * Height * Width (§3.6).")

Сохранено: /home/egor/lab_2_semester/python/AI/fishgrow/reports/figures/fig04_correlation.png

=== Интерпретация ===
Корреляции с целевой переменной (Weight):
Length3    0.923
Length2    0.919
Length1    0.916
Width      0.887
Height     0.724

Топ-3 пары признаков по |r|:
Length1  Length2    1.000
Length3  Length2    0.994
         Length1    0.992

Оговорка про нелинейные связи:
Корреляция Пирсона измеряет ЛИНЕЙНУЮ связь. Если связь нелинейная
(как Weight ~ Length здесь), коэффициент недооценивает силу связи.
Например, r(Weight, Length) ≈ 0.9, но реальная зависимость ближе
к степенной (W ~ L^3). Для линейной модели полезно логировать или
строить признак Volume = Length3 * Height * Width (§3.6).


/home/egor/lab_2_semester/python/AI/fishgrow/env/lib/python3.7/site-packages/ipykernel_launcher.py:15: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  from ipykernel import kernelapp as app


In [14]:
diff12 = (df["Length1"] - df["Length2"]).abs()
print("Length1 - Length2: max |diff| =", diff12.max())
print("Число строк, где Length1 == Length2:", (diff12 == 0).sum())
print("Число строк, где |diff| < 0.01:", (diff12 < 0.01).sum())
print()
print("Если max = 0 — поля идентичны, одно избыточно.")

Length1 - Length2: max |diff| = 4.399999999999999
Число строк, где Length1 == Length2: 0
Число строк, где |diff| < 0.01: 0

Если max = 0 — поля идентичны, одно избыточно.


In [15]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
sns.heatmap(df.isna(), cbar=False, cmap="viridis", ax=ax, yticklabels=False)
ax.set_title("Карта пропусков (жёлтый = пропуск, фиолетовый = значение)")
ax.set_xlabel("Столбец")
ax.set_ylabel("Наблюдение (индекс)")

quality_df = pd.DataFrame({
    "column": numeric_cols,
    "min": [df[c].min() for c in numeric_cols],
    "max": [df[c].max() for c in numeric_cols],
})
x = np.arange(len(quality_df))
ax = axes[1]
ax.bar(x - 0.2, quality_df["min"], width=0.4, label="min", color="lightsteelblue", edgecolor="black")
ax.bar(x + 0.2, quality_df["max"], width=0.4, label="max", color="steelblue", edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels(quality_df["column"], rotation=45)
ax.set_title("Минимум и максимум по числовым признакам")
ax.set_ylabel("Значение")
ax.legend()

fig.suptitle("Качество данных: пропуски и диапазоны", y=1.02)
plt.tight_layout()
save_fig(fig, "fig05_missing.png")
plt.show()

print("\n=== Интерпретация ===")
print(f"Всего ячеек: {df.size}")
print(f"Пропущено: {df.isna().sum().sum()}")
print(f"Доля пропусков: {df.isna().mean().mean()*100:.2f}%")
print("На левой панели все ячейки фиолетовые — пропусков нет.")
print("\nДиапазоны числовых признаков:")
print(quality_df.round(2).to_string(index=False))
print("\nВывод: пропусков в датасете нет. Диапазоны признаков реалистичные.")
print("Аномалия Weight = 0 (индекс 40) — разбирается в §3.5.")

Сохранено: /home/egor/lab_2_semester/python/AI/fishgrow/reports/figures/fig05_missing.png

=== Интерпретация ===
Всего ячеек: 1113
Пропущено: 0
Доля пропусков: 0.00%
На левой панели все ячейки фиолетовые — пропусков нет.

Диапазоны числовых признаков:
 column  min     max
 Weight 0.00 1650.00
Length1 7.50   59.00
Length2 8.40   63.40
Length3 8.80   68.00
 Height 1.73   18.96
  Width 1.05    8.14

Вывод: пропусков в датасете нет. Диапазоны признаков реалистичные.
Аномалия Weight = 0 (индекс 40) — разбирается в §3.5.


/home/egor/lab_2_semester/python/AI/fishgrow/env/lib/python3.7/site-packages/ipykernel_launcher.py:27: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.


In [16]:
col = "Weight"
q1, q3 = df[col].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
out_mask = (df[col] < lo) | (df[col] > hi)
n_out = int(out_mask.sum())

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.scatter(df.index, df[col], s=22, c="steelblue", label="наблюдения")
ax.scatter(df.index[out_mask], df[col][out_mask], s=55, c="red",
           label=f"выбросы по IQR: {n_out}", zorder=3)
ax.axhline(lo, color="gray", linestyle="--", label=f"Q1−1.5·IQR = {lo:.0f} г")
ax.axhline(hi, color="gray", linestyle=":",  label=f"Q3+1.5·IQR = {hi:.0f} г")
ax.axhline(0, color="orange", linestyle="-", linewidth=0.8,
           label="физическая нижняя граница (0 г)")
ax.set_title(f"Выбросы массы по правилу IQR (n = {len(df)})")
ax.set_xlabel("Индекс наблюдения")
ax.set_ylabel("Масса, г")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
save_fig(fig, "fig06_outliers.png")
plt.show()

print("\n=== Интерпретация ===")
print(f"Q1 = {q1:.0f} г, Q3 = {q3:.0f} г, IQR = {iqr:.0f} г")
print(f"Границы IQR: [{lo:.0f}; {hi:.0f}] г")
print(f"Выбросов: {n_out} из {len(df)}")
print("Индексы строк-выбросов:", df.index[out_mask].tolist())
print("\nЗначения выбросов:")
print(df.loc[out_mask, ["Species", "Weight", "Length3"]].to_string())
print("\nТакже отдельно: Weight = 0 на индексе 40 —")
print("это НЕ выброс, а ошибка данных (невозможная масса).")
print("Решение по нему — в §3.5.")

Сохранено: /home/egor/lab_2_semester/python/AI/fishgrow/reports/figures/fig06_outliers.png

=== Интерпретация ===
Q1 = 120 г, Q3 = 650 г, IQR = 530 г
Границы IQR: [-675; 1445] г
Выбросов: 3 из 159
Индексы строк-выбросов: [142, 143, 144]

Значения выбросов:
    Species  Weight  Length3
142    Pike  1600.0     64.0
143    Pike  1550.0     64.0
144    Pike  1650.0     68.0

Также отдельно: Weight = 0 на индексе 40 —
это НЕ выброс, а ошибка данных (невозможная масса).
Решение по нему — в §3.5.


/home/egor/lab_2_semester/python/AI/fishgrow/env/lib/python3.7/site-packages/ipykernel_launcher.py:22: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.


In [17]:
df["Volume_approx"] = df["Length3"] * df["Height"] * df["Width"]  # см³

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))

for sp, g in df.groupby("Species"):
    axes[0].scatter(g["Volume_approx"], g["Weight"], s=28, alpha=0.8, label=sp)
axes[0].set_title("Связь приближённого объёма и массы")
axes[0].set_xlabel("V = Length3 · Height · Width, см³")
axes[0].set_ylabel("Масса, г")
axes[0].legend(fontsize=8)

for sp, g in df.groupby("Species"):
    axes[1].scatter(g["Volume_approx"], g["Weight"], s=28, alpha=0.8, label=sp)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("То же в лог-лог шкале (проверка W ~ V^α)")
axes[1].set_xlabel("ln(V, см³)")
axes[1].set_ylabel("ln(масса, г)")
axes[1].legend(fontsize=8)

fig.suptitle("Гипотеза: масса пропорциональна объёму (V ≈ L3 · H · W)", y=1.02)
plt.tight_layout()
save_fig(fig, "fig07_volume_vs_weight.png")
plt.show()

print("\n=== Интерпретация ===")
# Защита от log(0): заменяем нули на NaN только для расчёта корреляции
valid = df[["Volume_approx", "Weight"]].replace(0, np.nan).dropna()
r_lin = valid.corr().iloc[0, 1]
lr = np.log(valid)
r_log = lr.corr().iloc[0, 1]
print(f"Корреляция Pearson (Volume, Weight): r = {r_lin:.3f}")
print(f"Корреляция Pearson в лог-шкале:       r = {r_log:.3f}")
print("\nПо видам (наклон log-log регрессии — оценка показателя степени α):")
for sp, g in df.groupby("Species"):
    x = np.log(g["Volume_approx"].clip(lower=1e-3))
    y = np.log(g["Weight"].clip(lower=1e-3))
    slope, intercept = np.polyfit(x, y, 1)
    print(f"  {sp:10s} α ≈ {slope:.2f}")
print("\nВывод: связь Volume ~ Weight сильная и близка к линейной (r > 0.95).")
print("В лог-шкале связь ещё теснее — масса растёт как примерно V^1.")
print("Виды различаются по плотности: при одинаковом объёме масса выше")
print("у Pike, ниже у Smelt. Гипотеза подтверждается, признак Volume")
print("имеет смысл как входной признак модели (детали — в §3.6).")

Сохранено: /home/egor/lab_2_semester/python/AI/fishgrow/reports/figures/fig07_volume_vs_weight.png

=== Интерпретация ===
Корреляция Pearson (Volume, Weight): r = 0.932
Корреляция Pearson в лог-шкале:       r = 0.991

По видам (наклон log-log регрессии — оценка показателя степени α):
  Bream      α ≈ 0.96
  Parkki     α ≈ 0.97
  Perch      α ≈ 0.98
  Pike       α ≈ 1.04
  Roach      α ≈ 1.31
  Smelt      α ≈ 0.76
  Whitefish  α ≈ 1.02

Вывод: связь Volume ~ Weight сильная и близка к линейной (r > 0.95).
В лог-шкале связь ещё теснее — масса растёт как примерно V^1.
Виды различаются по плотности: при одинаковом объёме масса выше
у Pike, ниже у Smelt. Гипотеза подтверждается, признак Volume
имеет смысл как входной признак модели (детали — в §3.6).


/home/egor/lab_2_semester/python/AI/fishgrow/env/lib/python3.7/site-packages/ipykernel_launcher.py:24: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.


In [19]:
# --- MAD-проверка выбросов (сравнение с IQR) ---
col = "Weight"
x = df[col].to_numpy()
med = np.median(x)
mad = np.median(np.abs(x - med))
z_mad = 0.6745 * (x - med) / mad  # устойчивая стандартизованная оценка

# Порог 3.5 — стандартный для z_MAD (соответствует ~3σ)
mask_mad = np.abs(z_mad) > 3.5

# IQR из графика 6
q1, q3 = df[col].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
mask_iqr = (df[col] < lo) | (df[col] > hi)

print("\n=== MAD-проверка ===")
print(f"Медиана: {med:.0f} г")
print(f"MAD:     {mad:.1f} г")
print(f"Порог |z_MAD| > 3.5")
print(f"Выбросов по MAD: {int(mask_mad.sum())}")
print(f"Индексы: {df.index[mask_mad].tolist()}")

print("\n=== Сравнение IQR и MAD ===")
print(f"Выбросы по IQR: {df.index[mask_iqr].tolist()}")
print(f"Выбросы по MAD: {df.index[mask_mad].tolist()}")
both = set(df.index[mask_iqr]) & set(df.index[mask_mad])
only_iqr = set(df.index[mask_iqr]) - set(df.index[mask_mad])
only_mad = set(df.index[mask_mad]) - set(df.index[mask_iqr])
print(f"\nОбщие: {sorted(both)}")
print(f"Только IQR:  {sorted(only_iqr)}")
print(f"Только MAD:  {sorted(only_mad)}")

# Отдельно: физически невозможные значения
bad_positive = df.index[df[col] <= 0].tolist()
print(f"\nФизически невозможные (Weight <= 0): {bad_positive}")


=== MAD-проверка ===
Медиана: 273 г
MAD:     204.0 г
Порог |z_MAD| > 3.5
Выбросов по MAD: 3
Индексы: [142, 143, 144]

=== Сравнение IQR и MAD ===
Выбросы по IQR: [142, 143, 144]
Выбросы по MAD: [142, 143, 144]

Общие: [142, 143, 144]
Только IQR:  []
Только MAD:  []

Физически невозможные (Weight <= 0): [40]


In [20]:
# --- Флаг для физически невозможных значений ---
df["weight_is_zero"] = (df["Weight"] <= 0).astype(int)
print("Строк с weight_is_zero = 1:", df["weight_is_zero"].sum())
print("Индексы:", df.index[df["weight_is_zero"] == 1].tolist())

Строк с weight_is_zero = 1: 1
Индексы: [40]
